In [ ]:
# Jason's implementation of directional algo
from algorithim import Params, Algorithm, Capon
from scipy.signal import find_peaks

import os
os.environ['DRJIT_LIBLLVM_PATH'] = '/usr/lib/x86_64-linux-gnu/libLLVM.so:'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import os # Configure which GPU

if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
    print(f"Using GPU {gpu_num}")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna
except ImportError as e:
    # Install Sionna if package is not already installed
    import os
    os.system("pip install sionna")
    import sionna

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
gpus = tf.config.list_physical_devices('GPU')
print(f"Found {len(gpus)} GPUs")
if gpus:
    print(f"Found GPU: {gpus[0].name}")
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

# Colab does currently not support the latest version of ipython.
# Thus, the preview does not work in Colab. However, whenever possible we
# strongly recommend to use the scene preview mode.
try: # detect if the notebook runs in Colab
    import google.colab
    no_preview = True # deactivate preview
except:
    if os.getenv("SIONNA_NO_PREVIEW"):
        no_preview = True
    else:
        no_preview = False

resolution = [480,320] # increase for higher quality of renderings

# Define magic cell command to skip a cell if needed
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)

# Set random seed for reproducibility
sionna.config.seed = 42

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera, AntennaArray, Antenna

# For link-level simulations
from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER
from sionna.ofdm import KBestDetector, LinearDetector
from sionna.mimo import StreamManagement
from sionna.ofdm import ResourceGrid

def calculate_heading_rad(start, end):
    # Calculate the difference vector
    diff = np.array(end) - np.array(start)
    
    # Calculate the heading angle in radians
    heading = np.arctan2(diff[1], diff[0])
    
    return heading

def get_rss_from_csi(csi):
  # Compute CSI power (power per subcarrier)
  csi_power = tf.abs(csi)**2

  # Compute RSS per receiver-transmitter pair (sum over subcarriers)
  rss_per_rx_tx = tf.reduce_sum(csi_power, axis=-1)  # Sum over subcarriers

  # Convert RSS from linear scale (e.g., Watts) to dBm
  rss_per_rx_tx_dBm = 10 * tf.math.log(rss_per_rx_tx / 1e-3) / tf.math.log(10.0)

  return rss_per_rx_tx_dBm

def dbm_to_watts(dbm):
    return 10. ** ((dbm-30)/10)

def watts_to_dbm(watts):
    epsilon = 1e-10  # Small constant to avoid log(0)
    return 10 * np.log10(watts + epsilon) + 30

# # Multiple Antennas like ASUS Router
def get_antenna_positions(spacing):
    return np.array([[0.0, spacing / 2 + spacing , 0.0],
                     [0.0, spacing / 2, 0.0],
                     [0.0, -spacing / 2, 0.0],
                     [0.0, -spacing / 2 - spacing, 0.0]])

def flip_trajectory(data):
    # Vector from origin to the first and last points of each trajectory
    first_points = data[:, 0, :]  # Shape (1000, 3)
    last_points = data[:, -1, :]  # Shape (1000, 3)

    # Compute the dot product of the first and last points with respect to the origin
    dot_products = np.sum(first_points * last_points, axis=1)

    # Identify trajectories heading away from the origin (dot product > 0)
    away_from_origin = dot_products < 0

    # Flip the trajectories that are not heading away from the origin
    data[~away_from_origin] = data[~away_from_origin, ::-1, :]

    return data

def path_loss_db(distance_m, frequency_hz):
    """
    Calculate free space path loss in dB.

    Parameters:
    - distance_m: distance between antennas in meters
    - frequency_hz: signal frequency in Hz

    Returns:
    - path loss in dB
    """
    c = 3e8  # speed of light in m/s
    loss_db = 20 * np.log10(distance_m) + 20 * np.log10(frequency_hz) - 147.55
    return loss_db

In [ ]:
# List for dataset
angles = []
angle_profile_values = []
rssi = []
rssi_normalized = []

SCENE_NAME = "meshes_512/small_scene0"
IMAGE_FOLDER = f"images/{SCENE_NAME}"

# os.environ['DRJIT_LIBLLVM_PATH'] = '/usr/lib/x86_64-linux-gnu/libLLVM.so:'
# os.environ['LD_LIBRARY_PATH'] = '/usr/lib/llvm-14/lib'
# Load integrated scene
scene = load_scene(f"models/{SCENE_NAME}.xml") # Try also sionna.rt.scene.etoile
# scene = load_scene(sionna.rt.scene.simple_wedge)

my_cam = Camera("my_cam", position=[10,0,30], look_at=[10,0,0])
scene.add(my_cam)

# Configure antenna array for all transmitters
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="V")

# To visualize the antenna array
# scene.tx_array.show()

# # Configure antenna array for all receivers
# scene.rx_array = PlanarArray(num_rows=1,
#                              num_cols=1,
#                              vertical_spacing=0.5,
#                              horizontal_spacing=0.5,
#                              pattern="iso",
#                              polarization="V")

antenna_positions = get_antenna_positions(0.03)

antenna_array_angle = -175 # Default

RelativeAntennas = AntennaArray(antenna=Antenna("dipole", "V"), 
                            positions=tf.Variable(antenna_positions.tolist()))

# scene.tx_array = RelativeAntennas 
scene.rx_array = RelativeAntennas

# Create transmitter
# tx_position = [-255., -140., 10.]
tx_position = [0., 0., 26.5 + 1] # Got this from ray intersection in path_on_mesh.ipynb
tx = Transmitter(name="tx",
                 position=tx_position,
                 orientation=[np.radians(0),0,0])

# Add transmitter instance to scene
scene.add(tx)

trajectories = np.load('small_mesh0.npy')[11:12,0:2,:]

total_traj = trajectories.shape[0]
seq_len = 20
ant_pairs = 4 # Example 4 rex x 1 tx or 4 rx x 4 tx
data = np.empty(shape=(0, seq_len - 1, 10 + seq_len * ant_pairs *52))
print(trajectories.shape)
trajectory_index = 0 # The one close to tx 2
d = np.linalg.norm(np.array(tx_position) - trajectories[trajectory_index, 0, :]) # meters
print(f"Distance from transmitter to trajectory {trajectory_index}: {d:.2f} m")
f =  5.745e9  # 2.4 GHz
pl_db = path_loss_db(d, f)
print(f"Path loss at {d}m and {f/1e9}GHz: {pl_db:.2f} dB")
print(f"Transmit power (dBm): {tx.power_dbm} dBm")
received_power_fspl = tx.power_dbm - pl_db
print(f"Received power at trajectory {trajectory_index} (dBm): {received_power_fspl:.2f} dBm")
num_steps = trajectories.shape[1]
trajectory_steps =  np.arange(num_steps - 1)# [10, 12, 14, 16, 18]

start_color = np.array([1, 0, 0])  # Red
end_color = np.array([0, 0, 1]) 
colors = [start_color + (end_color - start_color) * i / (num_steps - 1) for i in range(num_steps)]

# Calculate headings for the trajectory
calculated_headings = []
for i in trajectory_steps: #range(0, len(trajectories[0]) - 1):
    heading = calculate_heading_rad(trajectories[trajectory_index,i,:2], trajectories[trajectory_index,i + 1,:2])
    calculated_headings.append(heading)

rxs = []
for i, position in enumerate(trajectories[trajectory_index, np.arange(num_steps),:]):
    if i == (trajectories[0].shape[0] - 1):
        heading = calculated_headings[-1]
    else:
        heading = calculated_headings[i]
    rxs.append(Receiver(name=f"rx{i}",
              position=position,
              orientation=[heading,0,0]))    
    scene.add(rxs[i])

antenna_array_angle = np.radians(180) # heading

In [ ]:
scene.frequency = 2.14e9 #5.745 # in Hz; implicitly updates RadioMaterials

scene.synthetic_array = False # If set to False, ray tracing will be done per antenna element (slower for large arrays)
"When the property scene.synthetic_array is set to False, antenna arrays are explicitly modeled by finding paths between any pair of transmitting and receiving antennas in the scene. Otherwise, arrays are represented by a single antenna located in the center of the array. Phase shifts related to the relative antenna positions will then be applied based on a plane-wave assumption when the channel impulse responses are computed"

# Compute propagation paths
# sionna.rt.Scene.compute_paths(self, max_depth=3, method='fibonacci', num_samples=1000000, los=True, reflection=True, 
#                               diffraction=False, scattering=False, ris=True, scat_keep_prob=0.001, edge_diffraction=False, 
#                               check_scene=True, scat_random_phases=True, testing=False)
paths = scene.compute_paths(max_depth=2,
                            num_samples=1e5,  # Number of rays shot into directions defined
                                              # by a Fibonacci sphere , too few rays can
                                              # lead to missing paths
                            # method='fibonacci',  # Method to sample directions
                            los=True,  # Include Line-of-Sight paths
                            reflection=True,  # Include reflection paths
                            diffraction=True,  # Include diffraction paths
                            scattering=True,  # Include scattering paths
                            )

paths.normalize_delays = False

# Determine subcarrier frequencies
rg = ResourceGrid(num_ofdm_symbols=1,
                fft_size=52,
                dc_null = True,
                cyclic_prefix_length=20,
                #   pilot_pattern = "kronecker",
                #   pilot_ofdm_symbol_indices = [2, 8],
                subcarrier_spacing=5e6) #30e3)

frequencies = subcarrier_frequencies(rg.fft_size, rg.subcarrier_spacing)

# get rss
a, tau = paths.cir()

csi = cir_to_ofdm_channel(frequencies, a, tau, normalize=False)  # Non-normalized includes path-loss
print(f"csi.shape {csi.shape}")
csi_reshaped = []
for i in range(len(rxs)):
    csi_reshaped.append(np.array(csi[:,i,:,:,:,:,:]).reshape(4, 52))

# # Create a figure with 4 subplots
# fig, axs = plt.subplots(4, 1, figsize=(10, 8))

# # Plot each array in a separate subplot
# for j in range(len(rxs)):
#     for i in range(4):
#         axs[i].plot(csi_reshaped[j][i])
#         axs[i].set_title(f'Array {i+1}')
#         axs[i].set_xlabel('Frequency Index')
#         axs[i].set_ylabel('Value')


# Adjust layout
plt.tight_layout()
plt.show()

#Output: h_f ([batch size, num_rx, num_rx_ant, num_tx, num_tx_ant, num_time_steps, fft_size], tf.complex) – Channel frequency responses at frequencies
block_rss = get_rss_from_csi(csi).numpy()[0, :, 0, 0, 0, 0]
block_rss[np.isneginf(block_rss)] = -90
rssi.append(block_rss[:-1])
# Define range
min_rss = -90
max_rss = 0

# Normalize values to 0-1
normalized = (block_rss - min_rss) / (max_rss - min_rss)
rssi_normalized.append(normalized[:-1])

# Define three colors
color_black = np.array([0, 0, 0])
color_red = np.array([1, 0, 0])
color_blue = np.array([0, 0, 1])
colors = []
for value in normalized:
    if value <= 0.5:
        # Scale between black and gray
        color = color_black + (color_red - color_black) * (value / 0.5)
    else:
        # Scale between gray and white
        color = color_red + (color_blue - color_red) * ((value - 0.5) / 0.5)
    colors.append(color)

# Map normalized values to colors
# 0-1 range is split into 3 intervals: [0, 1/3), [1/3, 2/3), [2/3, 1]
# color_indices = np.digitize(normalized, bins=[1/3, 2/3], right=False)
# mapped_colors = [colors[idx] for idx in color_indices]

# for i, rx in enumerate(rxs):
#     rx.color = colors[i]


# Implementing Capon Algorithim
params = Params(antenna_positions[:, :2])

# Create a radial plot
plt.figure(figsize=(8, 8))
ax = plt.subplot(111, projection='polar')
angle_tmp = []
angle_profile_values_tmp = []
for i in range(0, len(rxs) - 1):
    
    csi_tmp = tf.reshape(csi[:,i,:,:,:,:,:], [4, 52]) #  tf.reshape(csi[:,i,:,:,:,:,:], [1, -1])

    AOA, theta_samples, profile_norm = Capon(params, 155, 80, csi_tmp).evaluate()

    # This for when going towards transmitter
    profile_norm = np.roll(profile_norm, 180)
    # Capture three top peaks from bottom of profile (90-270 degrees)
    evaluation = np.array([profile_norm[90:270]])
    evaluation = evaluation.flatten() 
    peaks, _ = find_peaks(evaluation)
    peak_values = evaluation[peaks]

    angle_profile_values_tmp.append(peak_values)
    angle_tmp.append(theta_samples[90:270][peaks])
    
    AOA = np.array([AOA])

    # Plot the evaluation profile
    ax.plot(theta_samples, profile_norm, label=f'Bearing {-np.degrees(AOA[0]):.2f} Deg.')

angles.append(angle_tmp)

angle_profile_values.append(angle_profile_values_tmp)

# Draw antenna array
# Rotation angle in radians for AoA algorithm
theta = 0 #antenna_array_angle # + 180)

# Rotation matrix for the z-plane
R_z = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta), np.cos(theta), 0],
    [0, 0, 1]
])

# Rotate antenna positions
rotated_positions = np.dot(antenna_positions, R_z.T)
dx = rotated_positions[:, :1]
dy = rotated_positions[:, 1:2]
angles_antenna = np.arctan2(dy, dx)  # Angles in radians
magnitudes = 5*np.sqrt(dx**2 + dy**2)  # Distances (optional for scaling)

# Boresight is perpendicular to the array_vector
array_vector = np.array([dx[-1] - dx[0], dy[-1] - dy[0]])
# Rotate 90 degrees (swap x and y, and negate one)
boresight_vector = np.array([-array_vector[1], array_vector[0]])
boresight_angle = np.arctan2(boresight_vector[1], boresight_vector[0])

# Plotting on a polar plot
ax.scatter(angles_antenna, magnitudes, c='r', label="Antenna Positions")  # Points
ax.plot(angles_antenna, magnitudes, 'b-', label="Array Direction") 
ax.plot([boresight_angle, boresight_angle], [0, .25], 'r-', label=f"Boresight Ground Truth {np.degrees(heading):.2f} Deg.")

# Add labels and legend
ax.set_title("Evaluation Profile in Radial Plot", va='bottom')
ax.set_theta_zero_location("N")  # Zero angle at the top (North)
ax.set_theta_direction(-1)  # Clockwise angle direction
ax.set_thetamin(-180) # only show top half
ax.set_thetamax(180)

# Show plot
plt.legend()
plt.show()

csi_reading = watts_to_dbm(tf.abs(csi.numpy().flatten()))

# scene.preview(show_orientations=True) # Use the mouse to focus on the visualized paths
scene.preview(paths, show_devices=True, show_paths=True, show_orientations=True) # Use the mouse to focus on the visualized paths

In [ ]:
import trimesh
mesh = trimesh.load_mesh("models/meshes_512/small_mesh0.ply")

ray_origins = (trajectories[0:1,0,:].squeeze() - np.array(tx_position))*np.ones((10,3)) * np.arange(10).reshape(10,1) / 10
ray_origins[ray_origins == -np.inf] = 0
ray_origins = np.concatenate((ray_origins, trajectories[0:1,0,:]), axis=0)
ray_origins[:,2] = -25
ray_directions = np.array([[0, 0, 1]]*ray_origins.shape[0])

locations, index_ray, index_tri = mesh.ray.intersects_location(
    ray_origins=ray_origins, ray_directions=ray_directions
)



import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

points = locations[:, :2]
print(points.shape)
print(points)
def fit_circle_scipy(points):
    x = points[:, 1]
    y = points[:, 0]

    def residuals(c):
        xc, yc, r = c
        return np.sqrt((x - xc)**2 + (y - yc)**2) - r

    x_m = np.mean(x)
    y_m = np.mean(y)
    r_m = np.mean(np.sqrt((x - x_m)**2 + (y - y_m)**2))
    c0 = np.array([x_m, y_m, r_m])

    result = least_squares(residuals, c0)
    return result.x

# Fit
xc, yc, r = fit_circle_scipy(points)
print(f"Center: ({xc:.2f}, {yc:.2f}), Radius: {r:.2f}")

# Plot
fig, ax = plt.subplots()
ax.scatter(points[:,1], points[:,0], label='Data Points')
ax.plot([points[0,1],points[10,1]], [points[0,0],points[10,0]], 'r-')

# circle = plt.Circle((xc, yc), r, color='r', fill=False, linewidth=2)
# ax.add_artist(circle)

# ax.set_aspect('equal', adjustable='datalim')
ax.plot()  

plt.ylim(-40, 10)
plt.legend()
plt.show()

radius = 5
spheres = []
for i in range(locations.shape[0]):
        sphere = trimesh.creation.icosphere(radius=radius, color=[1, 0, 0, 1])
        sphere.apply_translation(locations[i])
        spheres.append(sphere)

scene = trimesh.Scene([mesh, spheres])

scene.show()



In [ ]:
import numpy as np
### Calculation Check using Bertoni Chapter 7
# Path loss through knife edge
# Variables
R1 = 1000
R2 = 1000
R = R1 + R2
frequency = 1e9 #900e6  # 2.4 GHz
c = 3e8  # speed of light in m/s
wavelength = 3e8 / frequency
λ = 3e8 / frequency  # Example: wavelength (lambda)
k = 2 * np.pi / wavelength  # wavenumber
DT = 7.14 / np.sqrt(2 * np.pi * k) # Bertoni
L_knife = -10 * np.log10((λ * np.abs(DT) / (4 * np.pi * np.sqrt(R1 * R2 * R)))**2)
print(f"Path loss knife edge {L_knife}")

# Path loss through hill
Θ_hat = 0.0799
RH = 2500 # m radius of hill
D1 = (1 / (np.sqrt(2 * np.pi))) * (np.pi * RH / λ) ** (1/3) * 3.421 # 3.4.21 is the correction factor for TM polarization from Bertoni
# 1.019 came from Bertoni 7-11 TM polarization
ψ = (np.pi * RH / λ) ** (1/3) * np.exp(np.pi/6j).real * 1.019 # 1.019 is the correction factor for TM polarization from Bertoni

DT = np.abs(D1 * np.exp(-ψ*Θ_hat)) / np.sqrt(k)
L_round = -10 * np.log10((λ * np.abs(DT) / (4 * np.pi * np.sqrt(R1 * R2 * R)))**2)
print(f"Path loss round hill {L_round}")
print(L_knife - L_round)






In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# Your hill points
hill_points = np.array([
    [0., 0.],
    [-37.07094155, -3.44152746],
    [ -74.1418831, -6.88305492],
    [-111.21282465, -10.32458238],
    [-148.2837662, -13.76610984],
    [ -185.35470775, -17.20763729],
    [-222.4256493, -20.64916475],
    [-259.49659085, -24.09069221],
    [-296.5675324, -27.53221967],
    [-333.63847395, -30.97374713],
    [-370.7094155, -34.41527459]
])

# 2. Circle fitting
def fit_circle(points):
    x = points[:, 0]
    y = points[:, 1]

    def residuals(c):
        xc, yc, r = c
        return np.sqrt((x - xc)**2 + (y - yc)**2) - r

    x_m = np.mean(x)
    y_m = np.mean(y)
    r_m = np.mean(np.sqrt((x - x_m)**2 + (y - y_m)**2))
    c0 = np.array([x_m, y_m, r_m])

    result = least_squares(residuals, c0)
    return result.x  # (xc, yc, r)

xc, yc, R_H = fit_circle(hill_points)
print(f"Fitted circle center: ({xc:.2f}, {yc:.2f}), radius: {R_H:.2f} m")

# 3. Define transmitter and receiver
tx = hill_points[0]
rx = hill_points[-1]

# 4. Slope of Tx to Rx line
m_tx_rx = (rx[1] - tx[1]) / (rx[0] - tx[0])
print(f"Slope Tx-Rx line: {m_tx_rx:.4f}")

# 5. Find the tangent point (where hill slope matches Tx-Rx slope)
x = hill_points[:,0]
y = hill_points[:,1]
m_hill = np.diff(y) / np.diff(x)  # slope between each pair of points
slope_diff = np.abs(m_hill - m_tx_rx)
best_idx = np.argmin(slope_diff)

# Best tangent point is mid-point between best_idx and best_idx+1
hill_tangent_point = (hill_points[best_idx] + hill_points[best_idx+1]) / 2
print(f"Hill tangent point at: ({hill_tangent_point[0]:.2f}, {hill_tangent_point[1]:.2f})")

# 6. Elevation angle from Tx to tangent point
dx = hill_tangent_point[0] - tx[0]
dy = hill_tangent_point[1] - tx[1]
distance_horizontal = np.linalg.norm([dx, 0])
elevation_angle_rad = np.arctan2(-dy, distance_horizontal)  # careful: -dy because hill descends
elevation_angle_deg = np.degrees(elevation_angle_rad)
print(f"Elevation angle (tx to tangent point): {elevation_angle_deg:.2f} degrees")

# 7. Wireless parameters
frequency = 2.4e9  # 2.4 GHz
wavelength = 3e8 / frequency  # m
k = 2 * np.pi / wavelength  # wavenumber

theta_hat = 2 * elevation_angle_rad  # Total grazing angle (Tx + Rx)

print(f"Total grazing angle (radians): {np.degrees(theta_hat):.4f}")

# Simplified Bertoni model
D1 = 1.0  # Assume D1 ≈ 1
psi1_real = (k * R_H) ** (1/3)  # Rough approximation

exponent = -psi1_real * theta_hat
creeping_loss = D1 * np.exp(exponent)
creeping_loss_dB = -20 * np.log10(creeping_loss)

print(f"Creeping ray loss: {creeping_loss_dB:.2f} dB")

# 8. Plot everything
fig, ax = plt.subplots()
ax.scatter(hill_points[:,0], hill_points[:,1], label="Hill Points")

circle = plt.Circle((xc, yc), R_H, color='r', fill=False, linestyle='--', label="Fitted Circle")
ax.add_artist(circle)

ax.plot([tx[0], rx[0]], [tx[1], rx[1]], 'g--', label="Tx-Rx Line")
ax.plot(tx[0], tx[1], 'go', label="Transmitter")
ax.plot(rx[0], rx[1], 'bo', label="Receiver")
ax.plot(hill_tangent_point[0], hill_tangent_point[1], 'ro', label="Tangent Point", markersize=10)

ax.set_aspect('equal')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.legend()
plt.title("Hill Profile and Creeping Ray Tangent")
plt.grid(True)
plt.show()


In [ ]:
print(scene.radio_materials)
# Select an example object from the scene
so = scene.get('itu_concrete')
obj = scene.get("ground")
obj.radio_material.relative_permittivity = 1
obj.radio_material.scattering_coefficient = 1/np.sqrt(3)
from sionna.rt import LambertianPattern, DirectivePattern, BackscatteringPattern
obj.radio_material.scattering_pattern = LambertianPattern() #DirectivePattern(alpha_r=10)

print(obj.radio_material.name)
print(f"\nRadioMaterial:", obj.radio_material.conductivity.numpy())
print("Relative permittivity:", obj.radio_material.relative_permittivity.numpy())
print("Complex relative permittivity:", obj.radio_material.complex_relative_permittivity.numpy())
print("Relative permeability:", obj.radio_material.relative_permeability.numpy())
print("Scattering coefficient:", obj.radio_material.scattering_coefficient.numpy())
print("Scattering Pattern:", obj.radio_material.scattering_pattern)
print("XPD coefficient:", obj.radio_material.xpd_coefficient.numpy())

# # # Print name of assigned radio material for different frequenies
# for f in [3.5e9, 2.14e9]: # Print for differrent frequencies
#     # scene.frequency = f
#     print(f"\nRadioMaterial: {so.radio_material.name} @ {scene.frequency/1e9:.2f}GHz")
#     print("Conductivity:", so.radio_material.conductivity.numpy())
#     print("Relative permittivity:", so.radio_material.relative_permittivity.numpy())
#     print("Complex relative permittivity:", so.radio_material.complex_relative_permittivity.numpy())
#     print("Relative permeability:", so.radio_material.relative_permeability.numpy())
#     print("Scattering coefficient:", so.radio_material.scattering_coefficient.numpy())
#     print("XPD coefficient:", so.radio_material.xpd_coefficient.numpy())

In [ ]:
scene.frequency = 5.745e9 #5.745 # in Hz; implicitly updates RadioMaterials

scene.synthetic_array = False # If set to False, ray tracing will be done per antenna element (slower for large arrays)
"When the property scene.synthetic_array is set to False, antenna arrays are explicitly modeled by finding paths between any pair of transmitting and receiving antennas in the scene. Otherwise, arrays are represented by a single antenna located in the center of the array. Phase shifts related to the relative antenna positions will then be applied based on a plane-wave assumption when the channel impulse responses are computed"

# Compute propagation paths
# sionna.rt.Scene.compute_paths(self, max_depth=3, method='fibonacci', num_samples=1000000, los=True, reflection=True, 
#                               diffraction=False, scattering=False, ris=True, scat_keep_prob=0.001, edge_diffraction=False, 
#                               check_scene=True, scat_random_phases=True, testing=False)
paths = scene.compute_paths(max_depth=3,
                            num_samples=1e6,  # Number of rays shot into directions defined
                                              # by a Fibonacci sphere , too few rays can
                                              # lead to missing paths
                            method='fibonacci',  # Method to sample directions
                            
                            los=True,  # Include Line-of-Sight paths
                            reflection=True,  # Include reflection paths
                            diffraction=True,  # Include diffraction paths
                            scattering=True,  # Include scattering paths
                            edge_diffraction=False,  # Include edge diffraction paths
                            ris=False,  # Reflecting Intelligent Surfaces (RIS) are not considered in this example
                            scat_random_phases=True,  # Randomize phases of scattering paths
                            )

paths.normalize_delays = False

# Determine subcarrier frequencies
rg = ResourceGrid(num_ofdm_symbols=1,
                fft_size=52,
                dc_null = True,
                cyclic_prefix_length=20,
                #   pilot_pattern = "kronecker",
                #   pilot_ofdm_symbol_indices = [2, 8],
                subcarrier_spacing=5e6) #30e3)

frequencies = subcarrier_frequencies(rg.fft_size, rg.subcarrier_spacing)

# get rss
a, tau = paths.cir()

csi = cir_to_ofdm_channel(frequencies, a, tau, normalize=False)  # Non-normalized includes path-loss
print(f"csi.shape {csi.shape}")
csi_reshaped = []
for i in range(len(rxs)):
    csi_reshaped.append(np.array(csi[:,i,:,:,:,:,:]).reshape(4, 52))

# lets make a graph with CSI as time progresses (assuming time progresses when the Rx has moved/changed spatially) 
# we have csi_reshaped with dims (20, 4, 52)
# 20 refers to the time/space/each Rx, 4 is the num of Rx antennas, 52 is the different subcarrier freqs
# and the data stored itself is the complex number at Rx that .angle = phaseshift, .mag = amplitude -> dB
csi_reshaped = np.array(csi_reshaped)
print(csi_reshaped.shape)
# need this for amplitude to dB
def dB(x):
    return 10 * np.log10(np.abs(x) ** 2)
    # return 10 * np.log10(np.maximum(np.abs(x) ** 2, 1e-8))

# lets say we only care about Rx antenna #0
C = dB(csi_reshaped[:, 0, :])

# csi: shape (N_subcarriers, N_rx, N_tx), complex-valued
# |H|^2 gives power per subcarrier
csi_magnitude_squared = np.abs(csi_reshaped[0,:,:])**2
print(csi_magnitude_squared.shape)
power_per_subcarrier = csi_magnitude_squared.mean(axis=(0))  # average over Tx/Rx
avg_csi_power_db = 10 * np.log10(np.mean(power_per_subcarrier))
print(f"Average CSI Power (dB): {avg_csi_power_db:.2f} dB") 
print(f"Power {tx.power_dbm + avg_csi_power_db} dBm")  # Compare with received power

plt.pcolormesh(np.arange(csi_reshaped.shape[0]), np.arange(52), C.T)
plt.xticks(range(20))
plt.title(f"csi magnitudes as rx moves / time changes")
plt.ylabel("subcarrier index")
plt.xlabel("time / rx number")
plt.colorbar(orientation='vertical', fraction=0.02, pad=0.04, label="CSI Magnitude (dB)")

scene.preview(paths, show_devices=True, show_paths=True, show_orientations=True)


